# Cookbook: Foundry IQ End-to-End

Building production-grade AI systems requires more than simple vector search — you need retrieval that reasons, decomposes complex queries, and synthesizes answers with citations. **Foundry IQ** is Microsoft's intelligence layer that brings retrieval, reasoning, and synthesis together under a single, model-driven experience.

In this cookbook, you will:

- **Create a knowledge source** backed by an Azure AI Search index
- **Create a knowledge base** that pairs your data with an LLM for agentic retrieval
- **Query the knowledge base** and inspect synthesized answers with citations
- **Connect Foundry IQ to the Foundry Agent Service** so an agent can ground its responses in your data

## Prerequisites

"You can deploy all required resources automatically using the [Deploy to Azure](https://aka.ms/iq-series/deploytoazure) button — see the [Episode 1 README](../README.md#-deploy-azure-resources) for step-by-step instructions.\n",

If you prefer to set up resources manually, you need:

1. An **Azure AI Search** service in a [region that supports agentic retrieval](https://learn.microsoft.com/azure/search/search-region-support) with role-based access enabled.
2. A **Microsoft Foundry** project and resource — creating a project automatically provisions the resource.
3. Model deployments: a **text embedding model** (`text-embedding-3-large`) and a **chat model** (`gpt-4o-mini` or equivalent).
4. A **project connection** from your Foundry project to your Azure AI Search service.
5. Appropriate **RBAC roles** on your account: *Search Service Contributor*, *Search Index Data Contributor*, and *Search Index Data Reader* on the search service; *Cognitive Services User* on the Foundry resource for the search service's managed identity.

Create a `.env` file in this directory with your endpoint values:

```
SEARCH_ENDPOINT=https://<your-search-service>.search.windows.net
AOAI_ENDPOINT=https://<your-foundry-resource>.openai.azure.com
AOAI_EMBEDDING_MODEL=text-embedding-3-large
AOAI_EMBEDDING_DEPLOYMENT=text-embedding-3-large
AOAI_GPT_MODEL=gpt-4o-mini
AOAI_GPT_DEPLOYMENT=gpt-4o-mini
FOUNDRY_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
FOUNDRY_MODEL_DEPLOYMENT_NAME=gpt-4o-mini
AZURE_AI_SEARCH_CONNECTION_NAME=<your-search-connection-name>
FOUNDRY_PROJECT_RESOURCE_ID=/subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.MachineLearningServices/workspaces/<workspace>/projects/<project>
```

See the [agentic retrieval quickstart](https://learn.microsoft.com/azure/search/search-get-started-agentic-retrieval) for detailed setup instructions.

## Setup

Install required packages and load environment variables. We authenticate with `DefaultAzureCredential` so no API keys are stored in code.

> **Before you start:** Run `az login` in a terminal to sign in to Azure. This is required for `DefaultAzureCredential` to work.

In [1]:
%%capture
%pip install -U azure-search-documents==12.1.0b1 azure-ai-projects azure-identity python-dotenv

In [42]:
import logging
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

# Service endpoints
SEARCH_ENDPOINT = os.environ["SEARCH_ENDPOINT"]
AOAI_ENDPOINT = os.environ["AOAI_ENDPOINT"]

# Model deployments
EMBEDDING_MODEL = os.environ.get("AOAI_EMBEDDING_MODEL", "text-embedding-3-large")
EMBEDDING_DEPLOYMENT = os.environ.get("AOAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
GPT_MODEL = os.environ.get("AOAI_GPT_MODEL", "gpt-4o-mini")
GPT_DEPLOYMENT = os.environ.get("AOAI_GPT_DEPLOYMENT", "gpt-4o-mini")

# Resource names
INDEX_NAME = "earth-at-night"
KNOWLEDGE_SOURCE_NAME = "earth-knowledge-source"
KNOWLEDGE_BASE_NAME = "earth-knowledge-base"

credential = DefaultAzureCredential()
logging.getLogger("azure.search.documents._utils.model_base").setLevel(logging.ERROR)
print(f"Search endpoint: {SEARCH_ENDPOINT}")
print(f"Models: embedding={EMBEDDING_MODEL}, chat={GPT_MODEL}")

Search endpoint: https://stdagentvecstore.search.windows.net
Models: embedding=text-embedding-3-large, chat=gpt-5.4-mini


In [43]:
!az login --tenant 16b3c013-d300-468d-ac64-7eda0820b6d3

[
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "16b3c013-d300-468d-ac64-7eda0820b6d3",
    "id": "80ef7369-572a-4abd-b09a-033367f44858",
    "isDefault": true,
    "managedByTenants": [
      {
        "tenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47"
      },
      {
        "tenantId": "2f4a9838-26b7-47ee-be60-ccc1fdec5953"
      }
    ],
    "name": "MCAPS-Hybrid-REQ-39734-2022-babal",
    "state": "Enabled",
    "tenantId": "16b3c013-d300-468d-ac64-7eda0820b6d3",
    "user": {
      "name": "babal@microsoft.com",
      "type": "user"
    }
  },
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "16b3c013-d300-468d-ac64-7eda0820b6d3",
    "id": "5aa5ae5a-e1f6-4ed0-b06f-ddd3718184e8",
    "isDefault": false,
    "managedByTenants": [
      {
        "tenantId": "2f4a9838-26b7-47ee-be60-ccc1fdec5953"
      },
      {
        "tenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47"
      }
    ],
    "name": "MCAPS-Hybrid-REQ-42124-2022-marunachalam",
    "state": "Enabled",
 

We now have authenticated credentials and all configuration loaded from the environment. Nothing is hard-coded.

---

## Step 1 — Create a Search Index and Upload Documents

A knowledge source needs something to search over. We create a search index with text, vector, and semantic search capabilities, then load sample data from NASA's *Earth at Night* e-book. The key requirement for agentic retrieval is a `semantic_search` configuration with a `default_configuration_name`.

In [44]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=[
        SearchField(name="id", type="Edm.String", key=True, filterable=True),
        SearchField(name="page_chunk", type="Edm.String"),
        SearchField(
            name="page_embedding_text_3_large",
            type="Collection(Edm.Single)",
            stored=False,
            vector_search_dimensions=3072,
            vector_search_profile_name="hnsw_text_3_large",
        ),
        SearchField(name="page_number", type="Edm.Int32", filterable=True),
    ],
    vector_search=VectorSearch(
        profiles=[
            VectorSearchProfile(
                name="hnsw_text_3_large",
                algorithm_configuration_name="alg",
                vectorizer_name="azure_openai_text_3_large",
            )
        ],
        algorithms=[HnswAlgorithmConfiguration(name="alg")],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="azure_openai_text_3_large",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AOAI_ENDPOINT,
                    deployment_name=EMBEDDING_DEPLOYMENT,
                    model_name=EMBEDDING_MODEL,
                ),
            )
        ],
    ),
    # Semantic config is required for agentic retrieval
    semantic_search=SemanticSearch(
        default_configuration_name="semantic_config",
        configurations=[
            SemanticConfiguration(
                name="semantic_config",
                prioritized_fields=SemanticPrioritizedFields(
                    content_fields=[SemanticField(field_name="page_chunk")]
                ),
            )
        ],
    ),
)

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
index_client.create_or_update_index(index)
print(f"✅ Index '{INDEX_NAME}' ready")

✅ Index 'earth-at-night' ready


In [50]:
import requests
from azure.core.exceptions import HttpResponseError
from azure.search.documents import SearchClient

DATA_URL = "https://raw.githubusercontent.com/Azure-Samples/azure-search-sample-data/main/nasa-e-book/earth-at-night-json/documents.json"
response = requests.get(DATA_URL, timeout=30)
response.raise_for_status()
documents = response.json()

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=credential
)

upload_succeeded = False
try:
    results = []
    batch_size = 200
    for i in range(0, len(documents), batch_size):
        batch = documents[i : i + batch_size]
        results.extend(search_client.upload_documents(documents=batch))

    succeeded = sum(1 for r in results if r.succeeded)
    failed = [r for r in results if not r.succeeded]
    upload_succeeded = len(failed) == 0 and succeeded == len(results)

    print(f"Uploaded: {succeeded}/{len(results)} documents to '{INDEX_NAME}'")
    if failed:
        print(f"Failed documents: {len(failed)}")
        for item in failed[:5]:
            print(f"- key={item.key}, error={item.error_message}")

except HttpResponseError as exc:
    print(f"❌ Upload failed with status: {exc.status_code}")
    if exc.status_code in (401, 403):
        print("Your identity needs 'Search Index Data Contributor' on the Azure AI Search service to upload documents.")
        print("After assigning the role, rerun this cell.")
    else:
        print("Unexpected upload error. Check Search service health and index configuration.")

try:
    doc_count = search_client.get_document_count()
    print(f"Indexed document count: {doc_count}")
except HttpResponseError as exc:
    print(f"Could not read document count (status {exc.status_code}).")
    if exc.status_code in (401, 403):
        print("Your identity needs 'Search Index Data Reader' on the Azure AI Search service to query/count documents.")

if upload_succeeded:
    print("✅ Upload completed successfully.")

Uploaded: 194/194 documents to 'earth-at-night'
Indexed document count: 98
✅ Upload completed successfully.


We now have a search index populated with chunked text and vector embeddings. This is the raw material that Foundry IQ will reason over.

---

## Step 2 — Create a Knowledge Source

A **knowledge source** tells Foundry IQ *where* to find data. It points at our search index and declares which fields carry source metadata (useful for citations).

In [51]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    SearchIndexFieldReference,
)

knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE_NAME,
    description="NASA Earth at Night data",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        source_data_fields=[
            SearchIndexFieldReference(name="id"),
            SearchIndexFieldReference(name="page_number"),
        ],
    ),
)

index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
print(f"✅ Knowledge source '{KNOWLEDGE_SOURCE_NAME}' ready")

✅ Knowledge source 'earth-knowledge-source' ready


The knowledge source is now registered. It does not store data itself — it is a pointer that the knowledge base will use during retrieval.

---

## Step 3 — Create a Knowledge Base

A **knowledge base** wraps one or more knowledge sources with an LLM configuration. It defines *how* queries are planned, executed, and synthesized. We use `ANSWER_SYNTHESIS` mode so the LLM generates natural-language answers with inline citations instead of returning raw document chunks.

In [52]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import KnowledgeRetrievalOutputMode

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    models=[
        KnowledgeBaseAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=AOAI_ENDPOINT,
                deployment_name=GPT_DEPLOYMENT,
                model_name=GPT_MODEL,
            )
        )
    ],
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Provide a concise, informative answer grounded in the retrieved documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"✅ Knowledge base '{KNOWLEDGE_BASE_NAME}' ready")

✅ Knowledge base 'earth-knowledge-base' ready


At this point the full Foundry IQ pipeline is wired: **index → knowledge source → knowledge base → LLM**. We are ready to query.

---

## Step 4 — Query the Knowledge Base

Agentic retrieval goes beyond keyword matching. When you submit a query, the knowledge base:

1. Decomposes the query into focused subqueries
2. Runs them in parallel against the knowledge source
3. Reranks results with the semantic ranker
4. Synthesizes a grounded, cited answer

We pass a conversation-style `messages` list so the pipeline can use context across turns.

In [56]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalSemanticIntent,
    SearchIndexKnowledgeSourceParams,
 )

# Use GA retrieval directly to avoid preview endpoint mismatch errors in this environment.
retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint=SEARCH_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=credential,
    api_version="2026-04-01",
)

query = "What causes city lights to appear brighter from space during the holidays?"
result = retrieval_client.retrieve(
    retrieval_request=KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=query)],
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
                include_references=True,
                include_reference_source_data=True,
            )
        ],
    )
)
retrieval_mode = "ga-extractive"
print("✅ Retrieval complete (GA extractive)")

✅ Retrieval complete (GA extractive)


### Inspect the retrieval output

The response contains three key parts:
- **Response** — the extracted grounding data
- **Activity** — subqueries and planning steps (useful for debugging)
- **References** — source documents that ground the response

In [57]:
import json

if result is None:
    print("Skipping retrieval inspection because the request did not succeed.")
elif retrieval_mode and retrieval_mode.startswith("search-index-"):
    print("📝 Top passages from direct index fallback:\n")
    print(json.dumps(result, indent=2))
else:
    merged_text = "\n\n".join(
        content.text
        for resp in result.response
        for content in resp.content
    )

    # GA extractive mode returns JSON-encoded grounding data in response text.
    try:
        parsed = json.loads(merged_text)
        print("📝 Grounding data:\n", json.dumps(parsed, indent=2))
    except json.JSONDecodeError:
        print("📝 Answer:\n", merged_text)

    if result.activity:
        print("\n🔍 Query plan/activity:")
        print(json.dumps([a.as_dict() for a in result.activity], indent=2))

    if result.references:
        print(f"\n📚 {len(result.references)} reference(s) returned")

print(f"\nMode: {retrieval_mode}")

📝 Grounding data:
 [
  {
    "ref_id": 0,
    "content": "## Orbiting Tools to Observe Nightlights\n\n### Figure 1\n\nThis historical photograph, taken by an early crewed spacecraft, shows the curved limb of the Earth at night. The image looks down from orbit and displays closely clustered, bright city lights illuminating total darkness over the land. A slight scattering of clouds is visible in the atmosphere, their wispy forms barely lit by the city glow. The horizon appears at a diagonal, with the blackness of space to the side. Handwritten mission notation appears in black across the top of the image. This photograph demonstrates the early ability of orbital missions to capture nightlights from space, providing visual evidence of human activity across the Earth as seen from orbit.\n\n---\n\n<!-- PageFooter=\"Earth at Night\" -->\n<!-- PageNumber=\"10\" -->"
  },
  {
    "ref_id": 1,
    "content": "# Airglow\n\n## April 2, 2014\n\n### Locations shown: \n- Helsinki\n- St. Petersburg\

Notice how the knowledge base decomposed the question, retrieved relevant chunks, and produced a cited answer — all in a single API call. This is the core value of Foundry IQ: retrieval that *reasons*.

---

## Step 5 — Use with Foundry Agent Service

Foundry IQ becomes even more powerful when combined with the **Foundry Agent Service**. Instead of calling the retrieval API directly, you create an agent that uses the Knowledge Base’s **MCP (Model Context Protocol) endpoint** as a tool. The agent handles tool invocation, citation formatting, and multi-turn conversation.

The approach:
1. Build the Knowledge Base’s MCP endpoint URL
2. Create a **project connection** for MCP authentication
3. Create an agent with an `MCPTool` pointing at the Knowledge Base
4. Chat with the agent — it calls the Knowledge Base tool automatically and returns cited answers

This requires a Foundry project endpoint and project resource ID. Add these to your `.env`:

```
FOUNDRY_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
FOUNDRY_MODEL_DEPLOYMENT_NAME=gpt-4o
FOUNDRY_PROJECT_RESOURCE_ID=/subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.MachineLearningServices/workspaces/<workspace>/projects/<project>
```

In [58]:
# The Knowledge Base exposes an MCP endpoint for tool-based access
mcp_endpoint = f"{SEARCH_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version=2025-11-01-Preview"
print(f"✅ MCP endpoint: {mcp_endpoint}")

✅ MCP endpoint: https://stdagentvecstore.search.windows.net/knowledgebases/earth-knowledge-base/mcp?api-version=2025-11-01-Preview


### Create a project connection for MCP authentication

The agent communicates with the Knowledge Base via a `RemoteTool` project connection. This uses the project’s managed identity to authenticate against the search service’s MCP endpoint.

In [59]:
import requests
from azure.identity import get_bearer_token_provider

FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ.get("FOUNDRY_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")
PROJECT_RESOURCE_ID = os.environ["FOUNDRY_PROJECT_RESOURCE_ID"]
PROJECT_CONNECTION_NAME = "earth-kb-mcp-connection"

bearer_token_provider = get_bearer_token_provider(
    credential, "https://management.azure.com/.default"
)

conn_response = requests.put(
    f"https://management.azure.com{PROJECT_RESOURCE_ID}/connections/{PROJECT_CONNECTION_NAME}?api-version=2025-10-01-preview",
    headers={"Authorization": f"Bearer {bearer_token_provider()}"},
    json={
        "name": PROJECT_CONNECTION_NAME,
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "ProjectManagedIdentity",
            "category": "RemoteTool",
            "target": mcp_endpoint,
            "isSharedToAll": True,
            "audience": "https://search.azure.com/",
            "metadata": {"ApiType": "Azure"},
        },
    },
)
conn_response.raise_for_status()
print(f"✅ Project connection '{PROJECT_CONNECTION_NAME}' created")

✅ Project connection 'earth-kb-mcp-connection' created


### Create an agent with the MCP Knowledge Base tool

We connect to the Foundry project and create an agent with an `MCPTool` that points at the Knowledge Base’s MCP endpoint. The `project_connection_id` links the tool to the connection we just created, enabling managed-identity authentication.

In [60]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

project_client = AIProjectClient(
    endpoint=FOUNDRY_ENDPOINT, credential=credential
)

# Agent instructions optimized for knowledge retrieval with citations
instructions = """
You are a helpful assistant that must use the knowledge base to answer all the questions from user.
You must never answer from your own knowledge under any circumstances.
Every answer must always provide annotations for using the MCP knowledge base tool
and render them as: `【message_idx:search_idx†source_name】`
If you cannot find the answer in the provided knowledge base you must respond with "I don't know".
"""

# Create MCP tool with knowledge base connection
mcp_kb_tool = MCPTool(
    server_label="knowledge-base",
    server_url=mcp_endpoint,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=PROJECT_CONNECTION_NAME,
)

agent = project_client.agents.create_version(
    agent_name="earth-at-night-agent",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=instructions,
        tools=[mcp_kb_tool],
    ),
)

print(f"✅ Agent '{agent.name}' created (version={agent.version})")

✅ Agent 'earth-at-night-agent' created (version=1)


### Chat with the agent

We create a conversation session and send a query. The agent orchestrates calls to the MCP tool to retrieve content from the Knowledge Base, then synthesizes a cited answer.

In [61]:
openai_client = project_client.get_openai_client()

# Create a conversation session
conversation = openai_client.conversations.create()

# Send request that triggers the MCP tool
response = openai_client.responses.create(
    conversation=conversation.id,
    input="Why is the Phoenix nighttime street grid so sharply visible from space?",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

print("📝 Agent response:\n")
print(response.output_text)

📝 Agent response:

Phoenix is sharply visible from space at night because its metro area is laid out in a regular grid of streets and city blocks, and that grid is strongly lit. Major intersections, shopping centers, industrial sites, and corridors like Grand Avenue add bright lines and nodes, while the city’s outward spread and freeway network extend the pattern. The contrast with nearby dark areas like the Phoenix Mountains and agricultural fields makes the lit grid stand out even more.【5:0†source】【5:1†source】【5:2†source】


Here’s what happened under the hood:

1. The agent received the user question and invoked the `knowledge_base_retrieve` MCP tool
2. The tool sent search intents to the Knowledge Base’s MCP endpoint via the project connection
3. The Knowledge Base ran its agentic retrieval pipeline — query planning, hybrid search, semantic reranking
4. Results flowed back through MCP to the agent
5. The agent synthesized a cited answer from the retrieved content

This is the same MCP endpoint that **any MCP-compatible client** can use — GitHub Copilot, Claude Desktop, VS Code, or your own applications.

### Troubleshooting the agent + MCP connection

Foundry Agent Service can **mask** the underlying Azure AI Search error and surface a generic `tool_user_error` or `405 (Method Not Allowed)` during tool enumeration. These are almost always **auth/RBAC or configuration** issues, not transport bugs. If `responses.create(...)` fails:

**1. Assign the required roles** on the Azure AI Search service to the **Foundry project's managed identity** (keyless auth):

| Role | Assigned to | Purpose |
|---|---|---|
| `Search Index Data Reader` | Foundry project managed identity | Read index data during retrieval |
| `Search Service Contributor` | Foundry project managed identity | Resolve the knowledge base / MCP endpoint |

If the knowledge base uses an LLM for planning/synthesis, also grant the **Search service's** managed identity the `Cognitive Services User` role on the Azure OpenAI / Foundry resource.

**2. Keep both endpoints on the same project.** `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_PROJECT_RESOURCE_ID` must point to the **same** Foundry project, or the connection lookup fails with `Connection '...' not found`.

**3. Use the base resource endpoint for Azure OpenAI.** `AOAI_ENDPOINT` should be `https://<resource>.openai.azure.com` — do **not** append `/openai/v1`, which causes a `404` from the model endpoint.

**4. Isolate search vs. agent.** Query the knowledge base directly (the `retrieval_client.retrieve(...)` cell above, or the REST `retrieve` action). If the direct call succeeds, the problem is in how the agent authenticates to MCP (RBAC above). If it returns `401`, re-check the role assignments.

> Make sure your search service has **RBAC enabled** (Settings → Keys → *Both* or *Role-based access control*).


### Inspect the full response

The raw response contains metadata about tool calls, citations, and retrieval activity.

In [62]:
import json as _json

# Show the full response structure including tool calls and citations
print(_json.dumps(response.to_dict(), indent=2, default=str))

{
  "id": "resp_4dbc56974be90fc0006a1dcaa04da08190965eb9bd7aaf3ccd",
  "created_at": 1780337313.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "\nYou are a helpful assistant that must use the knowledge base to answer all the questions from user.\nYou must never answer from your own knowledge under any circumstances.\nEvery answer must always provide annotations for using the MCP knowledge base tool\nand render them as: `\u3010message_idx:search_idx\u2020source_name\u3011`\nIf you cannot find the answer in the provided knowledge base you must respond with \"I don't know\".\n",
  "metadata": {},
  "model": "gpt-5.4-mini",
  "object": "response",
  "output": [
    {
      "id": "mcpl_4dbc56974be90fc0006a1dcaa151f8819084e4a49a37e8fe63",
      "server_label": "knowledge-base",
      "tools": [
        {
          "input_schema": {
            "type": "object",
            "properties": {
              "queries": {
                "type": "array",
                "descri

### Clean up the agent and connection

In [63]:
# Delete the agent
project_client.agents.delete_version(
    agent_name=agent.name, agent_version=agent.version
)
print(f"✅ Agent '{agent.name}' version '{agent.version}' deleted")

# Delete the project connection
del_response = requests.delete(
    f"https://management.azure.com{PROJECT_RESOURCE_ID}/connections/{PROJECT_CONNECTION_NAME}?api-version=2025-10-01-preview",
    headers={"Authorization": f"Bearer {bearer_token_provider()}"},
)
del_response.raise_for_status()
print(f"✅ Project connection '{PROJECT_CONNECTION_NAME}' deleted")

✅ Agent 'earth-at-night-agent' version '1' deleted
✅ Project connection 'earth-kb-mcp-connection' deleted


---

## Clean Up

Remove the Foundry IQ resources we created. If you want to keep experimenting, skip this cell.

In [64]:
index_client.delete_knowledge_base(KNOWLEDGE_BASE_NAME)
print(f"Deleted knowledge base '{KNOWLEDGE_BASE_NAME}'")

index_client.delete_knowledge_source(knowledge_source=KNOWLEDGE_SOURCE_NAME)
print(f"Deleted knowledge source '{KNOWLEDGE_SOURCE_NAME}'")

index_client.delete_index(INDEX_NAME)
print(f"Deleted index '{INDEX_NAME}'")

Deleted knowledge base 'earth-knowledge-base'
Deleted knowledge source 'earth-knowledge-source'
Deleted index 'earth-at-night'


---

## Conclusion

In this cookbook we walked through the complete Foundry IQ lifecycle:

1. **Knowledge source** — We created a pointer to a search index so Foundry IQ knows *where* to look.
2. **Knowledge base** — We paired the knowledge source with an LLM, enabling query planning and answer synthesis.
3. **Agentic retrieval** — We queried the knowledge base and saw how it decomposes questions, retrieves in parallel, and returns cited answers.
4. **Agent Service integration** — We embedded the same search index into a Foundry agent, letting it ground responses automatically without manual retrieval code.

### Where to go next

- **Add more knowledge sources** — A single knowledge base can span multiple indexes and data types. Try adding a second index with different content.
- **Tune retrieval** — Experiment with `retrieval_reasoning_effort` levels and `answer_instructions` to control cost and quality.
- **Evaluate quality** — Use the [Azure AI Evaluation SDK](https://learn.microsoft.com/azure/ai-foundry/how-to/develop/evaluate-sdk) to measure groundedness and relevance at scale.
- **Explore the IQ Series** — Continue learning with [Episode 2: Data Pipelines](../../2-Foundry-IQ-Building-the-Data-Pipeline-with-Knowledge-Sources/) and [Episode 3: Multi-Source Queries](../../3-Foundry-IQ-Querying-the-Multi-Source-AI-Knowledge-Bases/).